# Phase 3 : Data Cleaning

## 1. Chargement des données et création d'une copie de travail

Cette phase consiste à préparer un jeu de données fiable pour
la segmentation des clients.

Les données originales seront conservées intactes. Toutes les
opérations de nettoyage seront effectuées sur une copie de travail.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Localisation du dossier contenant les données brutes
dossier_raw = Path("../data/raw")

if not dossier_raw.exists():
    dossier_raw = Path("data/raw")

# Recherche des fichiers Excel
fichiers_excel = list(dossier_raw.glob("*.xlsx"))

print("FICHIERS EXCEL TROUVÉS :")

for fichier in fichiers_excel:
    print(f"- {fichier.name} ({fichier.stat().st_size / (1024**2):.2f} Mo)")

FICHIERS EXCEL TROUVÉS :
- Copie de online_retail_II.xlsx (43.51 Mo)


In [3]:
# Lecture des deux feuilles du fichier Excel
feuilles = pd.read_excel(
    fichiers_excel[0],
    sheet_name=None
)

# Afficher les noms et dimensions des feuilles
for nom, donnees in feuilles.items():
    print(f"Feuille : {nom}")
    print(f"Dimensions : {donnees.shape}")
    print("-" * 40)

# Fusion des deux feuilles
df_brut = pd.concat(
    feuilles.values(),
    ignore_index=True
)

print("\nDIMENSIONS DU DATASET ORIGINAL")
print("Nombre de lignes :", len(df_brut))
print("Nombre de colonnes :", len(df_brut.columns))

display(df_brut.head())

Feuille : Year 2009-2010
Dimensions : (525461, 8)
----------------------------------------
Feuille : Year 2010-2011
Dimensions : (541910, 8)
----------------------------------------

DIMENSIONS DU DATASET ORIGINAL
Nombre de lignes : 1067371
Nombre de colonnes : 8


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
# Création d'une copie de travail
df = df_brut.copy()

print("Dataset original :", df_brut.shape)
print("Copie de travail :", df.shape)

print("\nLes données originales sont conservées :",
      df_brut.equals(df))

Dataset original : (1067371, 8)
Copie de travail : (1067371, 8)

Les données originales sont conservées : True


In [5]:
# Nombre de lignes avant suppression
lignes_avant = len(df)

# Suppression des doublons strictement identiques
df = df.drop_duplicates().copy()

# Nombre de lignes après suppression
lignes_apres = len(df)

print("NETTOYAGE DES DOUBLONS")
print("Lignes avant :", lignes_avant)
print("Lignes après :", lignes_apres)
print("Doublons supprimés :", lignes_avant - lignes_apres)
print("Pourcentage supprimé :",
      round((lignes_avant - lignes_apres) / lignes_avant * 100, 2), "%")

NETTOYAGE DES DOUBLONS
Lignes avant : 1067371
Lignes après : 1033036
Doublons supprimés : 34335
Pourcentage supprimé : 3.22 %


## 2. Suppression des doublons exacts

Les doublons strictement identiques ont été supprimés à l'aide
de la méthode `drop_duplicates()` de Pandas.

Cette opération a permis de supprimer 34 335 lignes,
soit 3,22 % du dataset initial.

Le dataset de travail contient désormais 1 033 036 lignes
et conserve les 8 variables initiales.

Les données brutes restent inchangées afin de préserver
une version de référence.

In [6]:
# Nombre et pourcentage de valeurs manquantes
valeurs_manquantes = pd.DataFrame({
    "Nombre": df.isnull().sum(),
    "Pourcentage": (df.isnull().mean() * 100).round(2)
})

display(valeurs_manquantes)

,Nombre,Pourcentage
Invoice,0,0.00
StockCode,0,0.00
Description,4275,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,235151,22.76
Country,0,0.00


In [7]:
# Séparer les transactions avec et sans identifiant client
transactions_sans_client = df[df["Customer ID"].isnull()]

transactions_avec_client = df[df["Customer ID"].notnull()]

print("TRANSACTIONS SANS IDENTIFIANT CLIENT")
print("Nombre :", len(transactions_sans_client))
print("Pourcentage :", round(
    len(transactions_sans_client) / len(df) * 100, 2
), "%")

print("\nTRANSACTIONS AVEC IDENTIFIANT CLIENT")
print("Nombre :", len(transactions_avec_client))
print("Pourcentage :", round(
    len(transactions_avec_client) / len(df) * 100, 2
), "%")

display(transactions_sans_client.head(10))

TRANSACTIONS SANS IDENTIFIANT CLIENT
Nombre : 235151
Pourcentage : 22.76 %

TRANSACTIONS AVEC IDENTIFIANT CLIENT
Nombre : 797885
Pourcentage : 77.24 %


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


In [8]:
# Analyse des transactions sans identifiant client
sans_client = df[df["Customer ID"].isna()].copy()

# Identifier les factures d'annulation
sans_client["Est_annulation"] = (
    sans_client["Invoice"].astype("string").str.startswith("C")
)

# Classifier les quantités
sans_client["Type_quantite"] = np.select(
    [
        sans_client["Quantity"] > 0,
        sans_client["Quantity"] < 0,
        sans_client["Quantity"] == 0
    ],
    [
        "Quantité positive",
        "Quantité négative",
        "Quantité nulle"
    ],
    default="Non déterminée"
)

# Classifier les prix
sans_client["Type_prix"] = np.select(
    [
        sans_client["Price"] > 0,
        sans_client["Price"] == 0,
        sans_client["Price"] < 0
    ],
    [
        "Prix positif",
        "Prix nul",
        "Prix négatif"
    ],
    default="Non déterminé"
)

print("RÉPARTITION DES QUANTITÉS")
display(sans_client["Type_quantite"].value_counts())

print("\nRÉPARTITION DES PRIX")
display(sans_client["Type_prix"].value_counts())

print("\nFACTURES D'ANNULATION")
display(sans_client["Est_annulation"].value_counts())

print("\nCOMBINAISON QUANTITÉ / PRIX")
display(pd.crosstab(
    sans_client["Type_quantite"],
    sans_client["Type_prix"]
))

RÉPARTITION DES QUANTITÉS


Type_quantite
Quantité positive    231045
Quantité négative      4106
Name: count, dtype: int64


RÉPARTITION DES PRIX


Type_prix
Prix positif    229202
Prix nul          5944
Prix négatif         5
Name: count, dtype: int64


FACTURES D'ANNULATION


Est_annulation
False    234437
True        714
Name: count, dtype: Int64


COMBINAISON QUANTITÉ / PRIX


Type_prix,Prix nul,Prix négatif,Prix positif
Type_quantite,,,
Quantité négative,3393,0,713
Quantité positive,2551,5,228489


In [9]:
# Transactions sans client avec quantité et prix positifs
ventes_sans_client = sans_client[
    (sans_client["Quantity"] > 0) &
    (sans_client["Price"] > 0)
]

print("Nombre de transactions positives sans client :",
      len(ventes_sans_client))

display(
    ventes_sans_client[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price", "Country"]
    ].head(15)
)

Nombre de transactions positives sans client : 228489


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Country
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,United Kingdom
1059,489548,22131,FOOD CONTAINER SET 3 LOVE HEART,2,2009-12-01 12:32:00,1.95,United Kingdom
1060,489548,22079,RIBBON REEL HEARTS DESIGN,10,2009-12-01 12:32:00,1.65,United Kingdom
1061,489548,22138,BAKING SET 9 PIECE RETROSPOT,3,2009-12-01 12:32:00,4.95,United Kingdom
1062,489548,22147,FELTCRAFT BUTTERFLY HEARTS,2,2009-12-01 12:32:00,1.45,United Kingdom


In [10]:
# Analyse des identifiants clients au niveau des factures

analyse_factures = df.groupby("Invoice").agg(
    nombre_clients=("Customer ID", "nunique"),
    lignes_sans_client=("Customer ID", lambda x: x.isna().sum()),
    lignes_avec_client=("Customer ID", lambda x: x.notna().sum())
)

# Factures contenant à la fois des lignes identifiées et non identifiées
factures_mixtes = analyse_factures[
    (analyse_factures["lignes_sans_client"] > 0) &
    (analyse_factures["lignes_avec_client"] > 0)
]

print("Nombre de factures distinctes :", len(analyse_factures))

print("Factures contenant des lignes avec et sans client :",
      len(factures_mixtes))

print("\nExemples :")
display(factures_mixtes.head(10))

Nombre de factures distinctes : 53628
Factures contenant des lignes avec et sans client : 0

Exemples :


,nombre_clients,lignes_sans_client,lignes_avec_client
Invoice,,,


In [11]:
# Séparation des transactions selon la disponibilité du Customer ID

df_sans_client = df[df["Customer ID"].isna()].copy()

df_clients = df[df["Customer ID"].notna()].copy()

print("Dataset après suppression des doublons :", len(df))
print("Transactions sans identifiant client :", len(df_sans_client))
print("Transactions avec identifiant client :", len(df_clients))

print("\nVérification :")
print("Total :", len(df_sans_client) + len(df_clients))
print("Séparation correcte :", 
      len(df_sans_client) + len(df_clients) == len(df))

Dataset après suppression des doublons : 1033036
Transactions sans identifiant client : 235151
Transactions avec identifiant client : 797885

Vérification :
Total : 1033036
Séparation correcte : True


### Traitement des identifiants clients manquants

Après analyse, 235 151 transactions (22,76 %) ne possèdent pas de
Customer ID.

Une vérification supplémentaire a été réalisée afin de déterminer
si certaines factures contenaient simultanément des lignes avec et
sans identifiant client. Sur les 53 628 factures distinctes, aucune
facture mixte n'a été détectée.

Il n'est donc pas possible de récupérer les identifiants manquants
à partir des autres lignes d'une même facture.

**Décision :**
- Les transactions sans Customer ID sont conservées séparément.
- Elles sont exclues de la future segmentation RFM, car elles ne
  peuvent pas être attribuées à un client identifiable.
- Les transactions avec Customer ID sont conservées pour la suite
  du projet.

Cette décision évite d'attribuer artificiellement des transactions
à des clients et de fausser les profils obtenus par clustering.

In [12]:
# Nombre de StockCode concernés par une description manquante

codes_sans_description = df.loc[
    df["Description"].isna(), "StockCode"
].unique()

print("Nombre de StockCode concernés :", len(codes_sans_description))

Nombre de StockCode concernés : 2451


### Traitement des descriptions manquantes

La variable Description contient 4 275 valeurs manquantes
après la suppression des doublons.

Cependant, cette variable n'est pas directement nécessaire
à notre objectif principal, qui consiste à segmenter les clients
à partir des indicateurs RFM.

Les valeurs manquantes sont donc conservées sans imputation
et sans suppression des transactions concernées.

La variable Description pourra être utilisée ultérieurement
pour enrichir l'interprétation commerciale des segments.

In [13]:
# Analyse des factures et des quantités négatives

print("Factures commençant par C :",
      df_clients["Invoice"].astype(str).str.startswith("C").sum())

print("Transactions avec quantité négative :",
      (df_clients["Quantity"] < 0).sum())

print("Transactions avec quantité nulle :",
      (df_clients["Quantity"] == 0).sum())

print("Transactions avec prix négatif :",
      (df_clients["Price"] < 0).sum())

print("Transactions avec prix nul :",
      (df_clients["Price"] == 0).sum())

Factures commençant par C : 18390
Transactions avec quantité négative : 18390
Transactions avec quantité nulle : 0
Transactions avec prix négatif : 0
Transactions avec prix nul : 70


In [14]:
# Création du dataset contenant uniquement les achats exploitables

df_achats = df_clients[
    (df_clients["Quantity"] > 0) &
    (df_clients["Price"] > 0) &
    (~df_clients["Invoice"].astype(str).str.startswith("C"))
].copy()

# Calcul du montant de chaque ligne
df_achats["Montant"] = df_achats["Quantity"] * df_achats["Price"]

print("Transactions initiales :", len(df_clients))
print("Transactions conservées :", len(df_achats))
print("Transactions exclues :", len(df_clients) - len(df_achats))
print("Nombre de clients :", df_achats["Customer ID"].nunique())
print("Montant total des achats :", round(df_achats["Montant"].sum(), 2))

Transactions initiales : 797885
Transactions conservées : 779425
Transactions exclues : 18460
Nombre de clients : 5878
Montant total des achats : 17374804.27


In [15]:
# Identifier les codes produits les plus fréquents
display(
    df_achats["StockCode"]
    .value_counts()
    .head(20)
)

# Examiner les descriptions contenant des opérations particulières
display(
    df_achats[
        df_achats["Description"].str.contains(
            "POSTAGE|BANK CHARGES|AMAZONFEE|MANUAL|DOT",
            case=False,
            na=False
        )
    ][["StockCode", "Description", "Quantity", "Price"]]
    .drop_duplicates()
    .head(20)
)

StockCode
85123A    5023
22423     3335
85099B    3296
84879     2692
20725     2609
21212     2557
47566     2098
20727     2045
22383     2039
21034     1950
22382     1935
21232     1935
22384     1874
21754     1852
22139     1848
22469     1821
20914     1821
20728     1820
84991     1813
POST      1803
Name: count, dtype: int64

,StockCode,Description,Quantity,Price
75,85014A,BLACK/BLUE DOTS RUFFLED UMBRELLA,3,5.95
76,85014B,RED/WHITE DOTS RUFFLED UMBRELLA,3,5.95
89,POST,POSTAGE,3,18.00
126,POST,POSTAGE,1,141.00
155,22199,FRYING PAN RED POLKADOT,4,4.25
173,POST,POSTAGE,1,130.00
304,84997D,PINK 3 PIECE MINI DOTS CUTLERY SET,4,3.75
305,84997C,BLUE 3 PIECE MINI DOTS CUTLERY SET,4,3.75
358,84997A,GREEN 3 PIECE MINI DOTS CUTLERY SET,3,3.75
377,84997B,RED 3 PIECE MINI DOTS CUTLERY SET,12,3.75


In [16]:
# Mesurer les lignes de frais de livraison
lignes_postage = df_achats["StockCode"].astype(str).str.upper() == "POST"

print("Lignes de livraison :", lignes_postage.sum())
print("Montant des frais de livraison :",
      round(df_achats.loc[lignes_postage, "Montant"].sum(), 2))

Lignes de livraison : 1803
Montant des frais de livraison : 124648.04


In [17]:
# Exclusion des frais de livraison
df_achats_final = df_achats[~lignes_postage].copy()

print("Transactions finales :", len(df_achats_final))
print("Clients finaux :", df_achats_final["Customer ID"].nunique())
print("Montant total :", round(df_achats_final["Montant"].sum(), 2))

# Vérification des valeurs
print("\nValeurs manquantes :")
display(df_achats_final.isnull().sum())

print("\nQuantités positives :", (df_achats_final["Quantity"] > 0).all())
print("Prix positifs :", (df_achats_final["Price"] > 0).all())

Transactions finales : 777622
Clients finaux : 5877
Montant total : 17250156.23

Valeurs manquantes :


Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Montant        0
dtype: int64


Quantités positives : True
Prix positifs : True


## Bilan du nettoyage

Les principales opérations de nettoyage réalisées sont :

1. Suppression des doublons exacts.
2. Séparation des transactions sans identifiant client.
3. Conservation des descriptions manquantes, non indispensables au RFM.
4. Exclusion des transactions avec quantités négatives (annulations et retours).
5. Exclusion des transactions avec prix nul.
6. Exclusion des frais de livraison.

Le dataset final contient uniquement les achats positifs associés à
des clients identifiables.

Il servira de base à la construction des indicateurs RFM et à la
segmentation des clients avec K-Means et CAH.

In [21]:
# Uniformiser les types des identifiants produits
df_achats_final["StockCode"] = (
    df_achats_final["StockCode"].astype("string")
)

# Vérifier les types
print(df_achats_final.dtypes)

Invoice                object
StockCode      string[python]
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
Montant               float64
dtype: object


In [22]:
# Enregistrer la base nettoyée au format Parquet
df_achats_final.to_parquet(
    chemin_base,
    index=False,
    engine="pyarrow"
)

print("Base nettoyée enregistrée avec succès !")
print("Emplacement :", chemin_base.resolve())

Base nettoyée enregistrée avec succès !
Emplacement : C:\Users\Lenovo\Documents\online-retail-ii-customer-segmentation\data\processed\online_retail_II_cleaned.parquet
